## 1. Setup and Configuration

In [1]:
import os
import gc
import datetime as dt
from pathlib import Path

import pandas as pd
import numpy as np
import pandas_datareader as pdr
from tqdm import tqdm

In [2]:
# =============================================================================
# DIRECTORY PATHS
# =============================================================================
CODE_DIR = Path(r"c:\Users\skazempour\Dropbox\Projects\42 - Machine learning from the crowd\Code")
DATA_DIR = Path(r"c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv")
FIGURES_DIR = Path(r"c:\Users\skazempour\Dropbox\Projects\42 - Machine learning from the crowd\Figures")

# =============================================================================
# INPUT/OUTPUT FOLDERS
# =============================================================================
INPUT_FOLDER = DATA_DIR / "feature_wo_messages"
OUTPUT_FOLDER = DATA_DIR / "feature_wo_messages_cleaned_mlcrowd"
OUTPUT_BY_YEAR_FOLDER = DATA_DIR / "cleaned_by_year_mlcrowd"

# Create output folders if they don't exist
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_BY_YEAR_FOLDER.mkdir(parents=True, exist_ok=True)

# =============================================================================
# PARAMETERS
# =============================================================================
# Date range for analysis
START_DATE = dt.date(2008, 1, 1)
END_DATE = dt.date(2024, 12, 31)

# Market hours (US/Eastern)
MARKET_OPEN = dt.time(9, 30, 0)
MARKET_CLOSE = dt.time(16, 0, 0)
PRE_MARKET_START = dt.time(4, 0, 0)
POST_MARKET_END = dt.time(20, 0, 0)

print(f"Input folder: {INPUT_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"Output by year folder: {OUTPUT_BY_YEAR_FOLDER}")

Input folder: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages
Output folder: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_cleaned_mlcrowd
Output by year folder: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\cleaned_by_year_mlcrowd


## 2. Inspect Raw Data Structure

In [3]:
# Get list of all CSV files
csv_files = sorted([f for f in os.listdir(INPUT_FOLDER) if f.endswith('.csv')])
print(f"Found {len(csv_files)} CSV files to process")

# Load first file to inspect structure
sample_file = INPUT_FOLDER / csv_files[0]
df_sample = pd.read_csv(sample_file, nrows=1000)

print(f"\nSample file: {csv_files[0]}")
print(f"Shape: {df_sample.shape}")
print(f"\nColumns: {list(df_sample.columns)}")
print(f"\nData types:\n{df_sample.dtypes}")
print(f"\nFirst few rows:")
df_sample.head()

Found 248 CSV files to process

Sample file: feature_wo_messages_000.csv
Shape: (1000, 7)

Columns: ['message_id', 'user_id', 'created_at', 'sentiment', 'parent_message_id', 'in_reply_to_message_id', 'symbol_list']

Data types:
message_id                  int64
user_id                     int64
created_at                 object
sentiment                 float64
parent_message_id         float64
in_reply_to_message_id    float64
symbol_list                object
dtype: object

First few rows:


,message_id,user_id,created_at,sentiment,parent_message_id,in_reply_to_message_id,symbol_list
0,4,593,2008-05-27T15:28:28Z,NaN,NaN,NaN,['V']
1,5,8687,2008-05-27T16:03:34Z,NaN,NaN,NaN,['NES']
2,6,549,2008-05-27T17:48:41Z,NaN,6.0,NaN,['AAPL']
3,7,170,2008-05-27T19:11:10Z,NaN,7.0,NaN,['XLE']
4,9,126,2008-05-27T22:39:09Z,NaN,NaN,NaN,['AAPL']


In [4]:
# Check sentiment distribution
print("Sentiment value counts:")
print(df_sample['sentiment'].value_counts(dropna=False))

# Check for missing values
print(f"\nMissing values:\n{df_sample.isna().sum()}")

Sentiment value counts:
sentiment
NaN    1000
Name: count, dtype: int64

Missing values:
message_id                   0
user_id                      0
created_at                   0
sentiment                 1000
parent_message_id          996
in_reply_to_message_id    1000
symbol_list                  0
dtype: int64


## 3. Build Trading Days Calendar

We use Fama-French data to identify trading days. Each message is mapped to its "first close after tweet" - the first market close following the message timestamp.

In [5]:
def build_trading_calendar(start_date, end_date):
    """
    Build a trading calendar using Fama-French data.
    Maps each calendar day to its first and second trading close.
    
    Parameters:
    -----------
    start_date : datetime.date
        Start of the date range
    end_date : datetime.date
        End of the date range
        
    Returns:
    --------
    pd.DataFrame with columns: date, business_day, first_close, second_close
    """
    print("Fetching Fama-French trading days...")
    
    # Download Fama-French data to get trading days
    ff = pdr.DataReader(
        'F-F_Research_Data_5_Factors_2x3_daily', 
        'famafrench',
        start=start_date,
        end=end_date
    )[0].reset_index()
    
    ff = ff.rename(columns={"Date": "first_close"})
    ff["second_close"] = ff["first_close"].shift(-1)
    
    # Create full calendar
    all_days = pd.date_range(start=start_date, end=end_date)
    trading_days = pd.DataFrame({"date": all_days})
    
    # Merge to identify trading days
    trading_days = pd.merge(
        trading_days, 
        ff[["first_close", "second_close"]], 
        left_on="date", 
        right_on="first_close", 
        how="left",
        indicator=True
    )
    
    trading_days["business_day"] = trading_days["_merge"] == "both"
    trading_days = trading_days[["date", "business_day", "first_close", "second_close"]]
    
    # Forward fill non-trading days to point to next trading day
    trading_days = trading_days.bfill()
    
    print(f"Trading calendar built: {trading_days['business_day'].sum()} trading days out of {len(trading_days)} total days")
    
    return trading_days

# Build the calendar
trading_calendar = build_trading_calendar(START_DATE, END_DATE)
trading_calendar.head(10)

Fetching Fama-French trading days...


C:\Users\skazempour\AppData\Local\Temp\ipykernel_64672\1242110481.py:20: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff = pdr.DataReader(


Trading calendar built: 4279 trading days out of 6210 total days


,date,business_day,first_close,second_close
0,2008-01-01,False,2008-01-02,2008-01-03
1,2008-01-02,True,2008-01-02,2008-01-03
2,2008-01-03,True,2008-01-03,2008-01-04
3,2008-01-04,True,2008-01-04,2008-01-07
4,2008-01-05,False,2008-01-07,2008-01-08
5,2008-01-06,False,2008-01-07,2008-01-08
6,2008-01-07,True,2008-01-07,2008-01-08
7,2008-01-08,True,2008-01-08,2008-01-09
8,2008-01-09,True,2008-01-09,2008-01-10
9,2008-01-10,True,2008-01-10,2008-01-11


## 4. Define Cleaning Functions

In [6]:
def parse_timestamp(df):
    """
    Parse created_at timestamp and convert to US/Eastern timezone.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with 'created_at' column
        
    Returns:
    --------
    pd.DataFrame with parsed timestamps
    """
    df = df.copy()
    
    # Parse timestamp
    df['created_at'] = pd.to_datetime(df['created_at'])
    
    # Convert to US/Eastern timezone (handle both timezone-aware and naive)
    try:
        df['created_at'] = df['created_at'].dt.tz_convert('US/Eastern').dt.tz_localize(None)
    except TypeError:
        # If timestamps are timezone-naive, assume UTC and convert
        df['created_at'] = df['created_at'].dt.tz_localize('UTC').dt.tz_convert('US/Eastern').dt.tz_localize(None)
    
    return df


def add_temporal_features(df):
    """
    Add temporal features needed for feature extraction.
    
    Features added:
    - calendar_date: Date portion of created_at
    - time: Time portion of created_at
    - hour: Hour of day (0-23)
    - is_after_hours: Boolean indicating if message is after market close
    - session: Market session category (pre_market, market_open, etc.)
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with parsed 'created_at' column
        
    Returns:
    --------
    pd.DataFrame with temporal features added
    """
    df = df.copy()
    
    # Extract date and time components
    df['calendar_date'] = pd.to_datetime(df['created_at'].dt.date)
    df['time'] = df['created_at'].dt.time
    df['hour'] = df['created_at'].dt.hour
    
    # After hours indicator (after 4 PM)
    df['is_after_hours'] = df['time'] > MARKET_CLOSE
    
    # Market session classification (for intraday features)
    def classify_session(t):
        h = t.hour
        m = t.minute
        if m < 30:
            m = 0
        else:
            m = 30
        return f"{h:02d}{m:02d}"
        
    df['session'] = df['time'].apply(classify_session)
    
    return df


def assign_trading_date(df, trading_calendar):
    """
    Assign each message to its first market close date.
    
    Logic:
    - Messages during market hours -> same day close
    - Messages after market close on trading days -> next trading day close
    - Messages on non-trading days -> next trading day close
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with 'calendar_date' and 'is_after_hours' columns
    trading_calendar : pd.DataFrame
        Trading calendar from build_trading_calendar()
        
    Returns:
    --------
    pd.DataFrame with 'date' column (trading date)
    """
    df = df.copy()
    
    # Merge with trading calendar
    df = pd.merge(
        df, 
        trading_calendar, 
        left_on='calendar_date', 
        right_on='date', 
        how='left',
        suffixes=('', '_cal')
    )
    
    # Assign trading date based on timing
    # Default: first close after the calendar date
    df['date'] = df['first_close']
    
    # If it's a trading day AND after market close -> use second close (next trading day)
    after_hours_on_trading_day = df['business_day'] & df['is_after_hours']
    df.loc[after_hours_on_trading_day, 'date'] = df.loc[after_hours_on_trading_day, 'second_close']
    
    # Clean up temporary columns
    cols_to_drop = ['calendar_date', 'first_close', 'second_close', 'date_cal']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=cols_to_drop)
    
    return df


def clean_dataframe(df, trading_calendar):
    """
    Full cleaning pipeline for a single DataFrame.
    
    Steps:
    1. Filter for valid sentiment (Bullish/Bearish)
    2. Parse timestamps to US/Eastern
    3. Add temporal features
    4. Assign trading date
    5. Drop unnecessary columns
    
    Parameters:
    -----------
    df : pd.DataFrame
        Raw StockTwits data
    trading_calendar : pd.DataFrame
        Trading calendar from build_trading_calendar()
        
    Returns:
    --------
    pd.DataFrame: Cleaned data
    """
    # Step 1: Filter for valid sentiment
    df_cleaned = df[df['sentiment'].isin(['Bullish', 'Bearish'])].copy()
    
    if len(df_cleaned) == 0:
        return df_cleaned
    
    # Step 2: Parse timestamps
    df_cleaned = parse_timestamp(df_cleaned)
    
    # Step 3: Add temporal features
    df_cleaned = add_temporal_features(df_cleaned)
    
    # Step 4: Assign trading date
    df_cleaned = assign_trading_date(df_cleaned, trading_calendar)

    # Step 5: Create weekend and holiday indicators
    df_cleaned['is_weekend'] = df_cleaned['created_at'].dt.weekday >= 5
    df_cleaned['is_holiday'] = (df_cleaned['business_day'] == False) & (df_cleaned['is_weekend'] == False)
    
    # Step 6: Drop columns not needed for feature extraction
    cols_to_drop = ['parent_message_id', 'in_reply_to_message_id']
    cols_to_drop = [c for c in cols_to_drop if c in df_cleaned.columns]
    df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    
    # Filter out rows with invalid dates
    df_cleaned = df_cleaned[df_cleaned['date'].notna()]
    
    return df_cleaned

## 5. Test Cleaning on Sample Data

In [7]:
# Test the cleaning pipeline on sample data
df_test = pd.read_csv(sample_file, nrows=10000)
df_cleaned_test = clean_dataframe(df_test, trading_calendar)

print(f"Original rows: {len(df_test):,}")
print(f"Cleaned rows: {len(df_cleaned_test):,}")
print(f"Retention rate: {len(df_cleaned_test)/len(df_test)*100:.1f}%")

print(f"\nCleaned columns: {list(df_cleaned_test.columns)}")
print(f"\nSession distribution:")
print(df_cleaned_test['session'].value_counts())

df_cleaned_test.head()

Original rows: 10,000
Cleaned rows: 1,462
Retention rate: 14.6%

Cleaned columns: ['message_id', 'user_id', 'created_at', 'sentiment', 'symbol_list', 'time', 'hour', 'is_after_hours', 'session', 'date', 'business_day', 'is_weekend', 'is_holiday']

Session distribution:
session
1830    621
1800    514
1900    254
1500     54
1430     11
1530      8
Name: count, dtype: int64


,message_id,user_id,created_at,sentiment,symbol_list,time,hour,is_after_hours,session,date,business_day,is_weekend,is_holiday
0,10000059,6472,2012-10-15 14:57:06,Bearish,"['ZNGA', 'META']",14:57:06,14,False,1430,2012-10-15,True,False,False
1,10000071,148519,2012-10-15 14:57:38,Bullish,['FVI'],14:57:38,14,False,1430,2012-10-15,True,False,False
2,10000072,75026,2012-10-15 14:57:39,Bullish,['GS'],14:57:39,14,False,1430,2012-10-15,True,False,False
3,10000084,155028,2012-10-15 14:58:20,Bullish,['WYNN'],14:58:20,14,False,1430,2012-10-15,True,False,False
4,10000088,75026,2012-10-15 14:58:23,Bullish,['JPM'],14:58:23,14,False,1430,2012-10-15,True,False,False


## 6. Process All Files

In [8]:
# Process all files and save cleaned versions
total_original_rows = 0
total_cleaned_rows = 0
processed_files = 0
failed_files = []

print("Processing all files...\n")

for csv_file in tqdm(csv_files):
    try:
        # Read the file
        input_path = INPUT_FOLDER / csv_file
        df = pd.read_csv(input_path)
        
        # Clean the dataframe
        df_cleaned = clean_dataframe(df, trading_calendar)
        
        # Save to output folder
        output_path = OUTPUT_FOLDER / csv_file
        df_cleaned.to_csv(output_path, index=False)
        
        # Update statistics
        total_original_rows += len(df)
        total_cleaned_rows += len(df_cleaned)
        processed_files += 1
        
        # Free memory
        del df, df_cleaned
        
    except Exception as e:
        failed_files.append((csv_file, str(e)))
        print(f"\nError processing {csv_file}: {e}")

gc.collect()

print(f"\n{'='*60}")
print(f"Processing complete!")
print(f"{'='*60}")
print(f"Files processed: {processed_files}/{len(csv_files)}")
print(f"Failed files: {len(failed_files)}")
print(f"Total original rows: {total_original_rows:,}")
print(f"Total cleaned rows: {total_cleaned_rows:,}")
if total_original_rows > 0:
    print(f"Retention rate: {total_cleaned_rows/total_original_rows*100:.2f}%")
print(f"\nOutput folder: {OUTPUT_FOLDER}")

Processing all files...



100%|██████████| 248/248 [16:28<00:00,  3.99s/it]


Processing complete!
Files processed: 248/248
Failed files: 0
Total original rows: 501,442,290
Total cleaned rows: 175,803,158
Retention rate: 35.06%

Output folder: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\feature_wo_messages_cleaned_mlcrowd


## 7. Estimate Memory Requirements for Full Load

In [9]:
# Get cleaned files
cleaned_files = sorted([f for f in os.listdir(OUTPUT_FOLDER) if f.endswith('.csv')])

# Sample files to estimate memory
SAMPLE_SIZE = min(5, len(cleaned_files))
sample_files = cleaned_files[:SAMPLE_SIZE]

print(f"Sampling {SAMPLE_SIZE} files to estimate memory usage...\n")

total_sample_rows = 0
total_sample_memory = 0

for csv_file in sample_files:
    file_path = OUTPUT_FOLDER / csv_file
    df_sample = pd.read_csv(file_path)
    
    memory_usage = df_sample.memory_usage(deep=True).sum()
    total_sample_rows += len(df_sample)
    total_sample_memory += memory_usage
    
    print(f"{csv_file}: {len(df_sample):,} rows, {memory_usage / (1024**2):.2f} MB")

# Calculate estimates
if total_sample_rows > 0:
    avg_memory_per_row = total_sample_memory / total_sample_rows
    estimated_memory = total_cleaned_rows * avg_memory_per_row
    
    print(f"\n{'='*60}")
    print(f"Memory Estimation:")
    print(f"{'='*60}")
    print(f"Average memory per row: {avg_memory_per_row:.2f} bytes")
    print(f"Total rows (cleaned): {total_cleaned_rows:,}")
    print(f"Estimated memory needed: {estimated_memory / (1024**3):.2f} GB")
    print(f"\nWith 50% overhead: {estimated_memory * 1.5 / (1024**3):.2f} GB")
    print(f"With 100% overhead: {estimated_memory * 2 / (1024**3):.2f} GB")

Sampling 5 files to estimate memory usage...

feature_wo_messages_000.csv: 566,254 rows, 179.17 MB
feature_wo_messages_001.csv: 571,472 rows, 180.85 MB
feature_wo_messages_002.csv: 576,380 rows, 182.40 MB
feature_wo_messages_003.csv: 605,965 rows, 191.79 MB
feature_wo_messages_004.csv: 603,319 rows, 190.97 MB

Memory Estimation:
Average memory per row: 331.85 bytes
Total rows (cleaned): 175,803,158
Estimated memory needed: 54.33 GB

With 50% overhead: 81.50 GB
With 100% overhead: 108.67 GB


## 8. Merge All Cleaned Files and Save by Year

This section loads all cleaned files, merges them, and saves by year for efficient access during feature extraction.

In [10]:
# Load all cleaned files
cleaned_files = sorted([f for f in os.listdir(OUTPUT_FOLDER) if f.endswith('.csv')])

print(f"Loading {len(cleaned_files)} cleaned files...")
print("This may take several minutes for large datasets...\n")

dfs = []
for i, csv_file in enumerate(tqdm(cleaned_files)):
    file_path = OUTPUT_FOLDER / csv_file
    df_temp = pd.read_csv(file_path)
    dfs.append(df_temp)
    
    # Progress update every 50 files
    if (i + 1) % 50 == 0:
        current_rows = sum(len(df) for df in dfs)
        print(f"  Loaded {i+1}/{len(cleaned_files)} files - {current_rows:,} rows")

print("\nConcatenating all dataframes...")
df_merged = pd.concat(dfs, ignore_index=True)

# Free memory
del dfs
gc.collect()

print(f"\n{'='*60}")
print(f"Merge Complete!")
print(f"{'='*60}")
print(f"Total rows: {len(df_merged):,}")
print(f"Total columns: {len(df_merged.columns)}")
print(f"Memory usage: {df_merged.memory_usage(deep=True).sum() / (1024**3):.2f} GB")
print(f"\nColumns: {list(df_merged.columns)}")
print(f"\nDate range: {df_merged['date'].min()} to {df_merged['date'].max()}")

Loading 248 cleaned files...
This may take several minutes for large datasets...



 20%|██        | 50/248 [00:26<01:49,  1.81it/s]

  Loaded 50/248 files - 30,063,820 rows


 40%|████      | 100/248 [00:55<01:30,  1.63it/s]

  Loaded 100/248 files - 67,551,298 rows


...

  Loaded 150/248 files - 109,542,843 rows


 81%|████████  | 200/248 [01:56<00:27,  1.75it/s]

  Loaded 200/248 files - 147,284,120 rows


100%|██████████| 248/248 [02:20<00:00,  1.76it/s]



Concatenating all dataframes...

Merge Complete!
Total rows: 175,803,158
Total columns: 13
Memory usage: 54.30 GB

Columns: ['message_id', 'user_id', 'created_at', 'sentiment', 'symbol_list', 'time', 'hour', 'is_after_hours', 'session', 'date', 'business_day', 'is_weekend', 'is_holiday']

Date range: 2010-06-02 to 2024-01-03


In [11]:
# Remove duplicates based on message_id
original_len = len(df_merged)
df_merged = df_merged.drop_duplicates(subset=['message_id'], keep='first')
duplicates_removed = original_len - len(df_merged)

print(f"Duplicates removed: {duplicates_removed:,} ({duplicates_removed/original_len*100:.2f}%)")
print(f"Final row count: {len(df_merged):,}")

Duplicates removed: 0 (0.00%)
Final row count: 175,803,158


In [12]:
# Convert date column to datetime and extract year
df_merged['date'] = pd.to_datetime(df_merged['date'])
df_merged['year'] = df_merged['date'].dt.year.astype(int)

# Get unique years
years = sorted(df_merged['year'].unique())
print(f"Years in data: {years}")
print(f"\nRows per year:")
print(df_merged['year'].value_counts().sort_index())

Years in data: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Rows per year:
year
2010       17104
2011       59196
2012      128436
2013      768054
2014     2078359
2015     3113930
2016     4769910
2017     9734088
2018    12658103
2019    11773627
2020    26222457
2021    62191431
2022    29254447
2023    13016673
2024       17343
Name: count, dtype: int64


In [13]:
# Save by year
print(f"\nSaving {len(years)} yearly files to: {OUTPUT_BY_YEAR_FOLDER}\n")

rows_written = 0
for year in years:
    df_year = df_merged[df_merged['year'] == year].drop(columns=['year'])
    output_path = OUTPUT_BY_YEAR_FOLDER / f"stocktwits_cleaned_{year}.csv"
    df_year.to_csv(output_path, index=False)
    rows_written += len(df_year)
    print(f"  {year}: {len(df_year):,} rows -> {output_path.name}")

print(f"\n{'='*60}")
print(f"Done! Total rows written: {rows_written:,}")
print(f"Output folder: {OUTPUT_BY_YEAR_FOLDER}")


Saving 15 yearly files to: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\cleaned_by_year_mlcrowd

  2010: 17,104 rows -> stocktwits_cleaned_2010.csv
  2011: 59,196 rows -> stocktwits_cleaned_2011.csv
  2012: 128,436 rows -> stocktwits_cleaned_2012.csv
  2013: 768,054 rows -> stocktwits_cleaned_2013.csv
  2014: 2,078,359 rows -> stocktwits_cleaned_2014.csv
  2015: 3,113,930 rows -> stocktwits_cleaned_2015.csv
  2016: 4,769,910 rows -> stocktwits_cleaned_2016.csv
  2017: 9,734,088 rows -> stocktwits_cleaned_2017.csv
  2018: 12,658,103 rows -> stocktwits_cleaned_2018.csv
  2019: 11,773,627 rows -> stocktwits_cleaned_2019.csv
  2020: 26,222,457 rows -> stocktwits_cleaned_2020.csv
  2021: 62,191,431 rows -> stocktwits_cleaned_2021.csv
  2022: 29,254,447 rows -> stocktwits_cleaned_2022.csv
  2023: 13,016,673 rows -> stocktwits_cleaned_2023.csv
  2024: 17,343 rows -> stocktwits_cleaned_2024.csv

Done! Total rows written: 175,803,158
Output folder: c:\Users\skazempour\Documents

## 9. Data Quality Summary

In [14]:
# Summary statistics for cleaned data
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

print(f"\n1. OVERALL STATISTICS")
print(f"   Total messages: {len(df_merged):,}")
# print(f"   Unique symbols: {df_merged['symbol'].nunique():,}")
print(f"   Unique users: {df_merged['user_id'].nunique():,}")
print(f"   Date range: {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")

print(f"\n2. SENTIMENT DISTRIBUTION")
sentiment_counts = df_merged['sentiment'].value_counts()
for sentiment, count in sentiment_counts.items():
    print(f"   {sentiment}: {count:,} ({count/len(df_merged)*100:.1f}%)")

print(f"\n3. SESSION DISTRIBUTION")
session_counts = df_merged['session'].value_counts()
for session, count in session_counts.items():
    print(f"   {session}: {count:,} ({count/len(df_merged)*100:.1f}%)")

# print(f"\n4. TOP 10 SYMBOLS BY VOLUME")
# top_symbols = df_merged['symbol'].value_counts().head(10)
# for symbol, count in top_symbols.items():
#     print(f"   {symbol}: {count:,}")

print(f"\n5. MISSING VALUES")
missing = df_merged.isna().sum()
for col, count in missing.items():
    if count > 0:
        print(f"   {col}: {count:,} ({count/len(df_merged)*100:.2f}%)")
if missing.sum() == 0:
    print("   No missing values in cleaned data")

print(f"\n" + "=" * 60)
print("Cleaning complete! Data is ready for feature extraction.")
print("=" * 60)

DATA QUALITY SUMMARY

1. OVERALL STATISTICS
   Total messages: 175,803,158
   Unique users: 1,170,927
   Date range: 2010-06-02 to 2024-01-03

2. SENTIMENT DISTRIBUTION
   Bullish: 150,835,163 (85.8%)
   Bearish: 24,967,995 (14.2%)

3. SESSION DISTRIBUTION
   930: 8,896,837 (5.1%)
   1000: 8,602,901 (4.9%)
   1030: 7,850,724 (4.5%)
   1100: 7,281,191 (4.1%)
   1530: 7,261,908 (4.1%)
   1130: 6,952,480 (4.0%)
   1500: 6,676,699 (3.8%)
   1200: 6,640,064 (3.8%)
   1230: 6,510,507 (3.7%)
   1430: 6,507,390 (3.7%)
   1400: 6,367,832 (3.6%)
   1300: 6,351,443 (3.6%)
   1330: 6,316,520 (3.6%)
   1600: 6,179,522 (3.5%)
   900: 5,423,930 (3.1%)
   1630: 4,597,246 (2.6%)
   830: 4,302,824 (2.4%)
   1700: 4,018,574 (2.3%)
   1730: 3,727,992 (2.1%)
   800: 3,533,830 (2.0%)
   1800: 3,483,156 (2.0%)
   1830: 3,270,711 (1.9%)
   1900: 3,119,661 (1.8%)
   1930: 3,088,074 (1.8%)
   2000: 2,907,613 (1.7%)
   730: 2,772,935 (1.6%)
   2030: 2,709,735 (1.5%)
   2100: 2,585,234 (1.5%)
   2130: 2,500,301 (